# Lecture 29 - Exploring SciPy
### EES 3350/5350 · Python in Earth Science · Friday, November 13, 2026 

SciPy is a huge library. No one memorizes it. What we do know is how to find a function, read its documentation, and test it on our own data in a few minutes. Today we're going to practice that skill.

**Today in 50 minutes**:
1. **the pattern** (\~10 min) : every SciPy function you'll ever use works the same three-step way
2. **stations** (\~25 min): your pair gets one SciPy function and some data you already know. Make it do something.
3. **share out** (\~10 min): each pair shows the class one figure and tells us when **you'd** reach for that function

You don't need to master your function. You need to know what it's **for**, what it takes **in**, and what it **gives back**, so you'll know to reach for it in your final project.

**Learning Objectives**:
- use the pattern: import a submodule -> call a function -> read what it gives back
- read a SciPy documentation page (Parameters, Returns, Examples)
- try a new function on real data, and change on argument at a time to see what it does


---
## Part 0 : set up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = "../../course-materials/"

---
## Part 1 : the pattern

You've already used SciPy. Earlier, it was `stats.linregress`. Using any other SciPy function works the same way:

|**step**|**`linregress`**|**in general**|
|--|--|--|
|1. import the **submodule**| `from scipy import stats`|`from scipy import SUBMODULE`|
|2. **call** the function on your data| `result = stats.linregress(x, y)`|`result = SUBMODULE.function(your data, settings...)`|
|3. **read** what came back|`result.slope`, `result.stderr`| an object, a number, an array, a tuple, **check!**|

Step 3 is where people get stuck. Different functions give back different kinds of things, and the only way to know is to look. So always start by asking Python:

In [ ]:
from scipy import stats

x = np.array([0, 1, 2, 3, 4])
y = np.array([1.1, 2.9, 5.2, 7.1, 8.8])

result = stats.linregress(x, y)

print(type(result))
result

##### `type()` told us...
- `linregress` gives back a special results object
##### printing it told us...
- that there is a name associated with each piece inside. **That's why `result.slope` worked earlier**.

#### Other functions give back:
- a plain number
- an array
- **two things at once**
- idk, you'll find out

### SciPy submodules
SciPy is split into **submodules** by the kind of job. A few that Earth scientists use all the time:

|**submodule**|**what it's for**|**an Earth science example**|
|--|--|--|
|`stats`| statistics, fits, tests, distributions | a trend with an uncertainty; is this site different from that one? |
|`optimize`| fitting **any** equaiton to data; finding minimums and roots | fit an exponential or a decay curve |
|`signal`| working with a series: peaks, smoothing, filtering | find every flood peak in a hydrograph |
|`interpolate` | estimating values **between** your measurements | fill a sparse survey into a fine grid |
|`integrate`| areas under curves, accumulation | flood volume under a hydrograph (**will cover on Monday**) |
|`ndimage`| operaiotns on grids and images | smooth or filter a DEM (**will cover in Week 13**) |

The full list is at https://docs.scipy.org/doc/scipy/reference/.

### Reading a documentation page
Every SciPy function has a page with the same sections. Open the one for `linregress` in a new tab: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html

|**section**|**what to look for**|
|--|--|
|the **signature** (the top line)| the function's name and its arguments, in order. Arguments with `=` have **defaults**, so you can leave them out. |
|**Parameters**| what each argument is, and what type it should be (array? number? function?) |
|**Returns**| **what comes back**. This is step 3!|
|**Examples**| working code at the bottom. Copy it, run it, then swap in your data.|

You can also see the documentation right in Jupyter. Put a `?` after the name and run the cell.

<center><b>Remember how I told you to make docstrings in functions you make?</b><p></center><center> You're essentially making the documentation page. docs.scipy.org just has a place where they're all published.</p></center>
 

In [ ]:
stats.linregress?

<div class="alert alert-success">

**Tip:** The **Examples** seciton is the fastest way into any few funciton. Run the example exactly as written first. Once it works, change one thing at a time until it runs on your data.
    
</div>

---
## Part 2 : stations

Your pair gets **one** station. Each station has:
- a short **goal** and a link to the function's documentation
- **starter code** that loads data you've seen before
- steps with `__` blanks, then an **explore** step where you change things and watch what happens

At the end, you'll show the class **one figure** and answer three questions about your function:
1. What does it **take in**?
2. What does it **give back**?
3. When would **you** reach for it?

#### Put your figure and the answers to your questions here: https://docs.google.com/presentation/d/1rrVN0Agh55mHNhpIuD43bI8C8v14LrnyAKRdaje-4cE/edit?usp=sharing

|**station**|**function**|**data**|
|--|--|--|
|A| `signal.find_peaks`| Mill Creek discharge, Feb-Apr 2021 (Lecture 25) |
|B| `optimize.curve_fit`| Mauna Loa CO$_2$ (Lectures 24 and 27) |
|C| `interpolate.CubicSpline`| Cedar River profile (Lecture 28) |
|D| `signal.savgol_filter`| Cedar River profile (Lecture 28) |

Finished early? Try a second station!



---
## Station A: `signal.find_peaks`, every flood in a hydrograph
**Goal**: find each storm peak in Mill Creek's record, without picking them out by eye.

**Docs**: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html

A **peak** is a point higher than its neighbors on both sides. That sounds simple, but a hydrograph has hundreds of tiny bumps. The trick is telling `find_peaks` which bumps are big enough to count.

In [ ]:
from scipy import signal

mill = pd.read_csv(data + "week11/usgs/millcreek_daily.csv")
mill["time"] = pd.to_datetime(mill["time"])
mill = mill.sort_values("time")

t = np.array(mill["time"])
q = np.array(mill["value"])

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t, q, color = "#0072B2")
ax.set_yscale("log")
ax.set_ylabel("daily mean discharge (ft$^3$/s)")
ax.set_title("Mill Creek at Thompson Lane, Feb-Apr 2021");

<div class="alert alert-info">

### A1 : call it, and read what comes back

`find_peaks` gives back **two things**, so we need to catch them in two variables. Figure out what to use as the input parameter(s). Then, look at what's in `peaks`.
    
</div>

In [ ]:
peaks, properties = signal.find_peaks(____)

print(type(peaks))
print(peaks)

<div class="alert alert-info">

### A2 : plot the peaks

Use the positions to pull out the peak dates and discharges (remember Lecture 14 indexing), and mark them on the hydrograph.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t, q, color="#0072B2")
ax.plot(t[____], q[____], "o", color="black", label="peak")

ax.set_yscale("log")
ax.set_ylabel("daily mean discharges (ft$^3$/s)")
ax.legend();

<div class="alert alert-info">

### A3 : explore, only the storms that matter

Open the documentation and find the `prominence` parameter. **Prominence** is how far a peak sticks up above the lower of the two valleys on either side of it.

1. Add `prominence=10` to `find_peaks` call and re-plot. Then try `100`, then `1000`. **How many peaks are left each time?** Answer visually and using `len()`.
2. Which value picks out what **you** would call separate storms?
3. What's in `properties` now? (`print(properties.keys())`)


</div>

In [ ]:
# your answer here

---
## Station B : `optimize.curve_fit`, fit any equation

**Goal:** fit an **exponential** curve to the CO$_2$ record, with uncertainty on every number.

**Docs:** https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html

In Lecture 24, a straight line left **U-shaped** residuals, because CO$_2$ is rising faster and faster. `np.polyfit` can only fit polynomials. `curve_fit` fits **any equation you can write as a function**.

Our model: CO$_2$ = a pre-industrial **baseline** $a$, plus an **excess** that grows exponentially:
$$ CO_2(t) = a + b\,e^{c(t-1959)}  $$

$b$ is the excess in 1959, and $c$ is how fast the excess grows (per year).

In [ ]:
from scipy import optimize

co2 = pd.read_csv(data + "week10/co2_maunaloa_annual.csv", header=None, names=["year", "ppm"])

t = np.array(co2["year"])
ppm = np.array(co2["ppm"])

def model(t, a, b, c):
    """Baseline 'a' plus an exponentially growing excess ('b' in 1959, growing at rate 'c' per year)."""
    return a + b * np.exp(c * (t-1959))

Notice how `model` is written: the $x$ values (`t`) come **first**, then the numbers to fit. `curve_fit` needs it in exactly that order.

<div class="alert alert-info">

### B1 : call it, and read what comes back

`curve_fit` needs three things: your **function**, the **x** data, and the **y** data. It also does muhc better with a starting guess, `p0=`, one guess per number to fit.

Guess $a$ ≈ 280 ppm (the pre-industrial level), $b$ ≈ 316 - 280 = 35 ppm, $c$ ≈ 0.02 per year.


</div>

In [ ]:
best, cov = optimize.curve_fit(____, t, ppm, p0= [____, ____, ____])

print(best)

<div class="alert alert-info">

### B2 : plot the fit and its residuals

Make a 2-panel figure: data + fitted curve on top, residuals (data - fit) on the bottom. Is the U from Lecture 24 gone?

</div>

In [ ]:
fit = model(t, *best)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(t, ppm, "o", markersize=3, color="gray", label="measured")
axes[0].plot(t, fit, color="black", label="exponential fit")
axes[0].set_ylabel("CO$_2$ (ppm)")
axes[0].legend()

axes[1].plot(t, ppm - ____, "o-", markersize=3)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("residual (ppm)")
axes[1].set_xlabel("year");

`model(t, *best)` is new: the `*` **unpacks** the array into three separate arguments, so it's the same as `model(t, best[0], best[1], best[2])` (like `**` for dictionaries in Lecture 10).

<div class="alert alert-info">

### B3 : explore, uncertainty on every number

1. The uncertainty (standard error) of each fitted number is `np.sqrt(np.diag(cov))`. Print each number ± 2 standard errors, like Lecture 27.
2. How long does the **excess** take the double? For exponential growth, the doubling time is $(ln2)/c$ (`np.log(2).c`).
3. Is your fitted $a$ close to 280 ppm? Try a bad starting guess, like `p0=[0, 1, 1]`. What happens?

</div>

In [ ]:
# your answer here

---
## Station C : `interpolate.CubicSpline`, filling in between measurements

**Goal:** pretend you only surveyed the Cedar River **once every kilometer**. How well can you rebuild the full 10-m profile form those few points?

**Docs:** https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.CubicSpline.html

**Interpolation** estimates values **between** points you measured. It's everywhere in Earth science: sparse wells into a water-table map, a few survey ponits  onto a channel, monthly samples onto daily dates. Here we have the full profile, so we can check how good the guess is.

In [ ]:
from scipy import interpolate

cedar = pd.read_csv(data + "week12/LowerCedarLongProfile.csv")
cedar = cedar[["riverkm", "bedelev.m", "wselev.m"]]
cedar = cedar.rename(columns={"bedelev.m": "bed_m", "wselev.m": "water_m"})
cedar = cedar.dropna()

x_all = np.array(cedar["riverkm"])
z_all = np.array(cedar["bed_m"])

# the "survey": every 100th point (100 points x 10 m = every 1 km)
x_survey = x_all[::100]
z_survey = z_all[::100]

print(len(x_all), "points in the full profile")
print(len(x_survey), "points in the survey")

<div class="alert alert-info">

### C1 : call it, and read what comes back

Make a spline from the survey points. What does `CubicSpline` give back?

</div>

In [ ]:
spline = interpolate.CubicSpline(____, ____)

print(type(spline))

> your answer here

<div class="alert alert-info">

### C2 : estimate the bed at all 10-m points, and compare

Call `spline` on the full set of distances, then plot the true profile, the survey points, and the spline. Zoom in to one stretch so you can see the difference.

</div>

In [ ]:
z_guess = spline(____)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(x_all, z_all, color="sienna", linewidth=0.8, label="true bed (every 10 m)")
ax.plot(x_survey, z_survey, "o", color="black", label="survey (every 1 km)")
ax,plot(x_all, z_guess, color="#0072B2", label="cubic spline")

ax.set_xlim(25, 18)
ax.set_ylim(60, 95)
ax.set_xlabel("distance upstream from the mouth (km)")
ax.set_label("elevation (m)")
ax.legend();

<div class="alert alert-info">

### C3 : explore, how sparse is too sparse?

1. How wrong is the spline on average? `np.mean(np.abs(z_guess - z_all))` is the average error in meters.
2. Change the survey spacing: try every 10th piont (100m), then every 500th (5 km). How does the error change?
3. Try it on the **water surface** (`water_m`) instead of the bed. Why is it so much easier to interpolate?
4. Compare with straight lines between points: `np.interp(x_all, x_survey, z_survey)` (NumPy's linear interpolation). Which has the smaller error?

</div>

In [ ]:
# your answer here

---
## Station D : `signal.savgol_filter`, smoothing and slope in one step

**Goal**: get the Cedar River's water-surface slope, but with one SciPy call instead of Wednesday's two steps (`np.gradient`, then `.rolling()`)

**Docs:** https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.savgol_filter.html

A **Savitsky-Golay filter** slides a window along the data, like `.rolling()`. But instead of taking hte mean in each window, it fits a small polynomial and keeps that curve's value, **or its slope**, at the middle. It's widely used for spectra, hydrographs, and profiles.

In [ ]:
from scipy import signal

cedar = pd.read_csv(data + "week12/LowerCedarLongProfile.csv")
cedar = cedar[["riverkm", "bedelev.m", "wselev.m"]]
cedar = cedar.rename(columns={"bedelev.m": "bed_m", "wselev.m": "water_m"})
cedar = cedar.dropna()

cedar["distance_m"] = cedar["riverkm"] * 1000

# wednesday's way: slope at every point, then a 1-km rolling average (in m/km)
slope = np.gradient(cedar["water_m"], cedar["distance_m"])
cedar["wed_m_per_km"] = pd.Series(slope, index=cedar.index).rolling(101, center=True).mean() * 1000

<div class="alert alert-info">

### D1 : smooth the profile

Start with smoothing only. `savgol_filter` needs the **data**, a **window length** (a number of points, and it must be **odd**), and a **polynomial order**. Use a 101-point window (about 1 km) and a straight line (order 1).

</div>

In [ ]:
z = np.array(cedar["water_m"])

z_smooth = signal.savgol_filter(z, window_length = ____, polyorder = ____)

print(type(z_smooth), z_smooth.shape, z.shape)

<div class="alert alert-info">

### D2 : get the slope directly

Now ask for the **derivative** instead, with `deriv=1`. It also needs `delta=`, the spacing between points, so it can divide by distance (just like `np.gradient`'s second argument). The ponts are 10 m apart. Convert to m/km and plot it against Wednesday's result.

</div>

In [ ]:
d = signal.savgol_filter(z, window_length=101, polyorder=2, deriv=___, delta=___)
cedar["savgol_m_per_km"] = d * 1000

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(cedar["riverkm"], cedar["wed_m_per_km"], color="gray", linewidth=3, label="Wednesday: gradient + rolling")
ax.plot(cedar["riverkm"], cedar["___"], color="#0072B2", label="Savitzky-Golay")

ax.invert_xaxis()
ax.set_xlabel("distance upstream from the mouth (km)")
ax.set_ylabel("slope (m drop per km)")
ax.legend();

<div class="alert alert-info">

### D3 : explore, what do the settings do?

1. Change `window_length` to 21, then 501. What happens to the slope curve? Why?
2. Change `polyorder` to 3 with the 101-point window. Does it follow the bumps more or less closely?
3. What happens if you forget `delta=`? (Look at the y-axis)
4. What happens with an **even** window, like 100?

</div>

In [ ]:
# your answer here

<div class="alert alert-danger">

**Common Error:** `savgol_filter` assumes the points are **evenly spaced** (that's why `delta` is one number). Where `dropna()` removed a few rows, the spacing isn't quite 10 m, so the result there is slightly off. `np.gradient` uses the real distances, so it doesn't have this problem. For unevenly spaced data, use `np.gradient`, or interpolate onto an even grid first (Station C!).

</div>

---
## Part 3 : share out

Each pair show **one figure** and answer the three questions. WHile you listen, fill in the last column for every station so you leave with the whole table.

|**function**|**takes in**|**gives back**|**I'd reach for it when...**|
|--|--|--|--|
|`stats.linregress`|`x`, `y`| a results object (`.slope`, `stderr`, ..)| I want one straight-line rate, with uncertainty|
|`signal.find_peaks`| a serires (+ settings like `prominence`)| two things: peak **positions**, and a dictionary of properties| |
|`optimize.curve_fit`| **my function**, `x`, `y`, a starting guess| two things: best-fit numbers, and a covarience matrix| |
|`interpolate.CubicSpline`| known `x`, known `y`| a **function** to call at new `x` values| |
|`signal.savgol_filter`| a series, window, order (+ `deriv`, `delta`)| an array, same length as the data| |

<div class="alert alert-warning">

### Question:
Your final project needs at least one result **with uncertainty**. Which of today's functions could give you one? how?
    
</div>

> your answer here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

- `linregress`: the standard error of the slope
- `curve_fit`: the standard error of **every** fitted number, from `np.sqrt(np.diag(cov))`
- `find_peaks`, `CubicSpline`, `savgol_filter`: not directly, but you can do a **sensitivity test: change the choice you made (`prominence`, survey spacing, window length) and report how much your answer changes. That counts as uncertainty for the final project too.

---
## A useful table

|**you want**|**write**|
|:--|:--|
|import a SciPy submodule|`from scipy import stats` (or `optimize`, `signal`, `interpolate`, ...)|
|see a function's documentation in Jupyter|`stats.linregress?`|
|what kind of thing did I get back?|`type(result)`|
|catch two things a function gives back|`a, b = some_function(...)`|
|peak positions in a series|`peaks, props = signal.find_peaks(y, prominence=P)`|
|fit any equation|`best, cov = optimize.curve_fit(f, x, y, p0=[...])`|
|standard errors from `curve_fit`|`np.sqrt(np.diag(cov))`|
|unpack an array into arguments|`f(x, *best)`|
|a smooth curve through points|`spline = interpolate.CubicSpline(x, y)`, then `spline(new_x)`|
|straight lines between points|`np.interp(new_x, x, y)`|
|smooth, or slope, in one step|`signal.savgol_filter(y, window, order, deriv=1, delta=dx)`|
